# MOFA+ Baseline — Multi-Omics Factor Analysis
**Person 1 (Mila) — Run this in Colab (no GPU needed)**

MOFA+ is the standard classical method for multi-omics integration.
We compare it against our foundation model approach.

**Runtime → Run all** — takes ~20-30 minutes for 800 samples.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q mofapy2 anndata numpy pandas scikit-learn xgboost umap-learn matplotlib seaborn

In [ ]:
SHARED_FOLDER_NAME = "multiomics-project"
from pathlib import Path
DRIVE_ROOT  = Path(f"/content/drive/MyDrive/{SHARED_FOLDER_NAME}")
METH_H5AD   = DRIVE_ROOT / "data/processed/tcga_methylation.h5ad"
RNA_H5AD    = DRIVE_ROOT / "data/processed/tcga_rna_seq.h5ad"
MANIFEST    = DRIVE_ROOT / "data/manifests/matched_samples.csv"
OUTPUT_DIR  = DRIVE_ROOT / "results/mofa"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

N_FACTORS = 30   # MOFA latent factors
N_HVG     = 2000 # top variable genes for RNA
N_HVM     = 5000 # top variable methylation probes

print('Methylation h5ad:', METH_H5AD.exists())
print('RNA h5ad        :', RNA_H5AD.exists())

In [ ]:
import numpy as np
import pandas as pd
import anndata as ad
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import label_binarize
import xgboost as xgb

print('Loading data...')
adata_meth = ad.read_h5ad(METH_H5AD)
adata_rna  = ad.read_h5ad(RNA_H5AD)
manifest   = pd.read_csv(MANIFEST)

print(f'Methylation: {adata_meth.shape}')
print(f'RNA-seq    : {adata_rna.shape}')

X_meth = adata_meth.X if isinstance(adata_meth.X, np.ndarray) else adata_meth.X.toarray()
X_rna  = adata_rna.X  if isinstance(adata_rna.X,  np.ndarray) else adata_rna.X.toarray()

# Select top variable features
top_meth = np.argsort(np.var(X_meth, axis=0))[-N_HVM:]
top_rna  = np.argsort(np.var(X_rna,  axis=0))[-N_HVG:]
X_meth_sel = X_meth[:, top_meth]  # (800, 5000)
X_rna_sel  = X_rna[:, top_rna]    # (800, 2000)

# Standardize
X_meth_sel = StandardScaler().fit_transform(X_meth_sel)
X_rna_sel  = StandardScaler().fit_transform(X_rna_sel)

labels = manifest['project'].values
le     = LabelEncoder()
y      = le.fit_transform(labels)

print(f'Cancer types: {dict(zip(*np.unique(labels, return_counts=True)))}')

In [ ]:
from mofapy2.run.entry_point import entry_point

print('Running MOFA+...')
ent = entry_point()

# MOFA expects list of views: [samples x features]
ent.set_data_options(scale_groups=False, scale_views=True)
ent.set_data_matrix(
    [[X_rna_sel], [X_meth_sel]],
    likelihoods=['gaussian', 'gaussian'],
    views_names=['RNA', 'Methylation'],
    groups_names=['TCGA']
)
ent.set_model_options(factors=N_FACTORS, spikeslab_weights=True, ard_factors=True, ard_weights=True)
ent.set_train_options(iter=1000, convergence_mode='fast', seed=42, verbose=False)
ent.build()
ent.run()

# Extract latent factors (the MOFA embedding)
mofa_embedding = ent.model.nodes['Z'].getExpectation()[0]  # (800, N_FACTORS)
np.save(OUTPUT_DIR / 'mofa_embedding.npy', mofa_embedding)
print(f'MOFA embedding shape: {mofa_embedding.shape}')

In [ ]:
# Evaluate MOFA embedding with XGBoost (same setup as foundation model experiments)
print('Evaluating MOFA embedding...')

def run_kfold(X, y, le, label):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    accs, f1s, aucs = [], [], []
    for train_idx, val_idx in skf.split(X, y):
        model = xgb.XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05,
                                   subsample=0.8, colsample_bytree=0.8,
                                   eval_metric='mlogloss', random_state=42, n_jobs=-1)
        model.fit(X[train_idx], y[train_idx])
        y_pred  = model.predict(X[val_idx])
        y_proba = model.predict_proba(X[val_idx])
        accs.append(accuracy_score(y[val_idx], y_pred))
        f1s.append(f1_score(y[val_idx], y_pred, average='macro', zero_division=0))
        y_bin = label_binarize(y[val_idx], classes=list(range(len(le.classes_))))
        aucs.append(roc_auc_score(y_bin, y_proba, multi_class='ovr', average='macro'))
    print(f'{label}: Acc={np.mean(accs):.3f}±{np.std(accs):.3f}  '
          f'F1={np.mean(f1s):.3f}±{np.std(f1s):.3f}  '
          f'AUC={np.mean(aucs):.3f}±{np.std(aucs):.3f}')
    return {'label': label, 'acc': np.mean(accs), 'f1': np.mean(f1s), 'auc': np.mean(aucs)}

# Also evaluate raw PCA baselines
from sklearn.decomposition import PCA
pca_rna  = PCA(n_components=128, random_state=42).fit_transform(X_rna_sel)
pca_meth = PCA(n_components=128, random_state=42).fit_transform(X_meth_sel)

results = []
results.append(run_kfold(mofa_embedding,                   y, le, 'MOFA+ (30 factors)'))
results.append(run_kfold(pca_rna,                          y, le, 'PCA RNA (128 dims)'))
results.append(run_kfold(pca_meth,                         y, le, 'PCA Methylation (128 dims)'))
results.append(run_kfold(np.hstack([pca_rna, pca_meth]),   y, le, 'PCA Concat (256 dims)'))

pd.DataFrame(results).to_csv(OUTPUT_DIR / 'baseline_metrics.csv', index=False)
print('\nSaved to:', OUTPUT_DIR / 'baseline_metrics.csv')

In [ ]:
# UMAP of MOFA factors colored by cancer type
import umap
import matplotlib.pyplot as plt

CANCER_COLORS = {'TCGA-BRCA':'#e41a1c','TCGA-LUAD':'#377eb8','TCGA-COAD':'#4daf4a',
                 'TCGA-KIRC':'#ff7f00','TCGA-LIHC':'#984ea3','TCGA-THCA':'#a65628'}

reducer   = umap.UMAP(n_components=2, random_state=42, n_neighbors=15)
embedding = reducer.fit_transform(mofa_embedding)

fig, ax = plt.subplots(figsize=(8, 7))
for cancer in np.unique(labels):
    mask = labels == cancer
    ax.scatter(embedding[mask,0], embedding[mask,1],
               c=CANCER_COLORS.get(cancer,'#333'), label=cancer.replace('TCGA-',''),
               s=25, alpha=0.8, edgecolors='none')
ax.set_title('UMAP of MOFA+ Factors (30)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.set_aspect('equal','datalim')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'umap_mofa.png', dpi=150, bbox_inches='tight')
plt.show()
print('Done! Compare this UMAP to Geneformer/MethylGPT — foundation models should be cleaner.')